# HMM 기반 S&P500 시장 레짐 판단 모델 (v2 — 전면 개편)
### 딥러닝 프로젝트 — STEP 1: HMM 정답지(Target Label) 생성

**목표**: 거시경제+기술지표로 향후 20일 시장 레짐을 분류하는 Target Y 생성
**개편 내용**: Multi-Variant HMM 비교 · Soft Posterior 저장 · 정량 이벤트 F1


---
## 0. Setup & Constants

In [ ]:
# hsmmlearn 은 설치 실패 가능 → try/except 로 폴백
!pip install fredapi hmmlearn finance-datareader statsmodels -q
try:
    import subprocess
    subprocess.run(['pip', 'install', 'hsmmlearn', '-q'], check=True)
    print("hsmmlearn 설치 성공")
except Exception as e:
    print(f"hsmmlearn 설치 실패 (HSMM 폴백 모드로 진행): {e}")


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
import datetime
import math
import json
import os

from fredapi import Fred
import FinanceDataReader as fdr
from hmmlearn import hmm

from scipy import stats
from scipy.stats import jarque_bera, skew, kurtosis, probplot, shapiro
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score

try:
    from hsmmlearn import hsmm as hsmmlib
    HSMM_AVAILABLE = True
    print("hsmmlearn 로드 성공")
except Exception:
    HSMM_AVAILABLE = False
    print("hsmmlearn 없음 → HSMM 구간 폴백(duration 분포 분석)으로 진행")

warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 100

# ── 전역 상수 ──
RANDOM_STATE  = 42
START_DATE    = '1999-01-01'
END_DATE      = datetime.datetime.today().strftime('%Y-%m-%d')
FUTURE_WINDOW = 20
N_SEEDS       = 10
STATE_RANGE   = range(2, 8)

np.random.seed(RANDOM_STATE)
print(f"분석 기간 : {START_DATE} ~ {END_DATE}")
print(f"미래 참조 윈도우: {FUTURE_WINDOW}일 | 초기값 탐색: {N_SEEDS}개")


In [ ]:
# Google Colab 시크릿에서 API 키 로드
from google.colab import userdata
FRED_API_KEY = userdata.get('FRED_API_KEY')
fred = Fred(api_key=FRED_API_KEY)


---
## 1. Data Loading (SPY + FRED ALFRED)

In [ ]:
def load_sp500(start: str, end: str) -> pd.DataFrame:
    """
    SPY ETF 기반 주가 피처 생성.
    HMM 전용 피처  : log_return, vol_5d/20d/60d, vol_ratio
    딥러닝 Track1  : sma_gap_*, rsi_14, macd_pct, macd_hist
    """
    raw = fdr.DataReader('SPY', start, end)
    df  = pd.DataFrame(index=raw.index)

    df['log_return']    = np.log(raw['Adj Close'] / raw['Adj Close'].shift(1))
    df['vol_5d']        = df['log_return'].rolling(5).std()
    df['vol_20d']       = df['log_return'].rolling(20).std()
    df['vol_60d']       = df['log_return'].rolling(60).std()
    df['log_vol_5d']    = np.log(df['vol_5d'])
    df['log_vol_20d']   = np.log(df['vol_20d'])
    df['log_vol_60d']   = np.log(df['vol_60d'])
    df['vol_ratio_5_20'] = df['vol_5d'] / df['vol_20d']
    df['vol_ratio_5_60'] = df['vol_5d'] / df['vol_60d']

    for w in [20, 60, 200]:
        df[f'sma_gap_{w}d'] = raw['Adj Close'] / raw['Adj Close'].rolling(w).mean() - 1

    delta    = raw['Adj Close'].diff()
    gain     = delta.where(delta > 0, 0)
    loss     = -delta.where(delta < 0, 0)
    rs       = gain.ewm(alpha=1/14, adjust=False).mean() / loss.ewm(alpha=1/14, adjust=False).mean()
    df['rsi_14']     = 100 - (100 / (1 + rs))
    ema12            = raw['Adj Close'].ewm(span=12, adjust=False).mean()
    ema26            = raw['Adj Close'].ewm(span=26, adjust=False).mean()
    macd             = ema12 - ema26
    df['macd_pct']   = macd / raw['Adj Close']
    df['macd_hist']  = df['macd_pct'] - macd.ewm(span=9, adjust=False).mean() / raw['Adj Close']
    df['close']      = raw['Adj Close']

    return df.dropna()


stock_df = load_sp500(START_DATE, END_DATE)
print(f"주가 데이터: {stock_df.index.min().date()} ~ {stock_df.index.max().date()} ({len(stock_df):,}일)")
stock_df.head()


In [ ]:
def safe_fred_fetch(fred_client, ticker):
    try:
        return fred_client.get_series_first_release(ticker)
    except Exception as e:
        print(f"[{ticker}] ALFRED 오류 → 일반 데이터 대체 ({e})")
        return fred_client.get_series(ticker)


def safe_reindex(data, target_index):
    """합집합 인덱스 → ffill → 평일 추출 (발표지연 반영)"""
    if isinstance(data, pd.Series):
        data = pd.to_numeric(data, errors='coerce')
    else:
        data = data.apply(pd.to_numeric, errors='coerce')
    combined = data.index.union(target_index)
    return data.reindex(combined).ffill().loc[target_index]


def load_macro_features(fred_client, start, end):
    market_days = pd.bdate_range(start=start, end=end)
    frames = {}

    m2 = pd.to_numeric(safe_fred_fetch(fred_client, 'M2SL'), errors='coerce')
    m2_yoy = m2.pct_change(12) * 100; m2_yoy.name = 'M2_YOY'
    m2_yoy.index += pd.DateOffset(days=25)
    frames['M2_YOY'] = safe_reindex(m2_yoy, market_days)

    nfci = fred_client.get_series('NFCI'); nfci.name = 'NFCI'
    nfci.index += pd.DateOffset(days=5)
    frames['NFCI'] = safe_reindex(nfci, market_days)

    ls = fred_client.get_series('T10Y3M'); ls.name = 'YIELD_SPREAD'
    frames['YIELD_SPREAD'] = safe_reindex(ls, market_days)

    cpi = pd.to_numeric(safe_fred_fetch(fred_client, 'CPIAUCSL'), errors='coerce')
    cpi_yoy = cpi.pct_change(12) * 100; cpi_yoy.name = 'CPI_YOY'
    cpi_yoy.index += pd.DateOffset(days=15)
    frames['CPI_YOY'] = safe_reindex(cpi_yoy, market_days)

    de = safe_fred_fetch(fred_client, 'NCBCMDPMVCE'); de.name = 'DEBT_EQUITY'
    de.index += pd.DateOffset(days=75)
    frames['DEBT_EQUITY'] = safe_reindex(de, market_days)

    dep   = safe_fred_fetch(fred_client, 'DPSACBW027SBOG')
    loans = safe_fred_fetch(fred_client, 'TOTLL')
    ltdr  = pd.DataFrame({'deposit': dep, 'loans': loans})
    ltdr.index += pd.DateOffset(days=9)
    ltdr  = safe_reindex(ltdr, market_days)
    ltdr_ratio = (ltdr['loans'] / ltdr['deposit']); ltdr_ratio.name = 'LOAN_DEPOSIT'
    frames['LOAN_DEPOSIT'] = ltdr_ratio

    macro_df = pd.concat(frames, axis=1)
    print(f"매크로 수집 완료: {macro_df.shape[1]}개 지표")
    return macro_df


macro_df   = load_macro_features(fred, START_DATE, END_DATE)
common_idx = stock_df.index.intersection(macro_df.index)
total_df   = pd.concat([stock_df.loc[common_idx], macro_df.loc[common_idx]], axis=1).dropna()

print(f"병합 완료: {total_df.shape} | {total_df.index.min().date()} ~ {total_df.index.max().date()}")
assert total_df.isna().sum().sum() == 0, "NaN 존재 — dropna 확인 필요"
total_df.head()


---
## 2. Feature Engineering (Track1 + Track2)

정규성 검사(JB/SW/KS)로 HMM 가우시안 가정 위반 정도를 확인한다.

In [ ]:
def normality_report(df: pd.DataFrame) -> pd.DataFrame:
    """피처별 정규성 검사 (JB / SW / KS)"""
    results = []
    for col in df.columns:
        x = df[col].dropna().values
        sk  = skew(x); ku = kurtosis(x)
        jb_stat, jb_p = jarque_bera(x)
        sw_stat, sw_p = (shapiro(x[:5000]) if len(x) > 5000 else shapiro(x))
        x_std = (x - x.mean()) / x.std()
        ks_stat, ks_p = stats.kstest(x_std, 'norm')
        fat_tail = abs(ku) > 3
        reject_count = sum([jb_p < 0.05, sw_p < 0.05, ks_p < 0.05])
        verdict = 'Non-Gaussian' if reject_count >= 2 else 'Gaussian'
        results.append({'Feature': col, 'Skewness': round(sk, 3),
                        'Kurt(excess)': round(ku, 3), 'Fat-Tail': fat_tail,
                        'JB_p': round(jb_p, 4), 'SW_p': round(sw_p, 4),
                        'KS_p': round(ks_p, 4), 'Reject(2/3)': reject_count,
                        'Verdict': verdict})
    return pd.DataFrame(results).set_index('Feature')


TRACK1_FEATS = ['log_return', 'vol_5d', 'vol_20d', 'vol_60d',
                'vol_ratio_5_20', 'vol_ratio_5_60',
                'sma_gap_20d', 'sma_gap_60d', 'sma_gap_200d',
                'rsi_14', 'macd_pct', 'macd_hist']
TRACK2_FEATS = ['M2_YOY', 'NFCI', 'YIELD_SPREAD',
                'CPI_YOY', 'DEBT_EQUITY', 'LOAN_DEPOSIT']

analysis_cols = TRACK1_FEATS + TRACK2_FEATS
norm_df       = normality_report(total_df[analysis_cols])
print("정규성 검사 결과:")
print(norm_df.to_string())
print(f"\n비정규: {(norm_df['Verdict']=='Non-Gaussian').sum()}개 | Fat-tail: {norm_df['Fat-Tail'].sum()}개")


---
## 3. Stationarity Diagnostics (ADF + KPSS)

비정상 피처(`DEBT_EQUITY`, `YIELD_SPREAD` 등)는 `*_diff` 컬럼으로 차분 보존.
**HMM 입력에는 사용하지 않음. STEP 2 딥러닝 입력 시 이 컬럼을 사용할 것.**


In [ ]:
def stationarity_diagnostics(df: pd.DataFrame, features: list) -> pd.DataFrame:
    """ADF + KPSS 결과 표 반환. verdict: Stationary / Non-Stationary / Conflicting"""
    results = []
    for col in features:
        x = df[col].dropna()
        adf_stat, adf_p, _, _, _, _ = adfuller(x)
        try:
            kpss_stat, kpss_p, _, _ = kpss(x, regression='ct', nlags='auto')
        except Exception:
            kpss_stat, kpss_p = np.nan, np.nan

        adf_ok  = adf_p < 0.05
        kpss_ok = (kpss_p if not np.isnan(kpss_p) else 1) >= 0.05

        if adf_ok and kpss_ok:
            verdict = 'Stationary'
        elif not adf_ok and not kpss_ok:
            verdict = 'Non-Stationary'
        else:
            verdict = 'Conflicting'

        results.append({'Feature': col,
                        'ADF_p': round(adf_p, 4),
                        'KPSS_p': round(kpss_p, 4) if not np.isnan(kpss_p) else np.nan,
                        'Verdict': verdict})
    return pd.DataFrame(results).set_index('Feature')


stat_df = stationarity_diagnostics(total_df, analysis_cols)
print("정상성 검사 결과:")
print(stat_df.to_string())
print(f"\n정상: {(stat_df['Verdict']=='Stationary').sum()} | "
      f"비정상: {(stat_df['Verdict']=='Non-Stationary').sum()} | "
      f"상충: {(stat_df['Verdict']=='Conflicting').sum()}")


In [ ]:
# 비정상 피처를 1차 차분하여 *_diff 컬럼 추가
# HMM 입력에는 사용 안 함. STEP 2 딥러닝 입력 권장.
NON_STATIONARY_FEATS = stat_df[stat_df['Verdict'] == 'Non-Stationary'].index.tolist()
print(f"비정상 피처: {NON_STATIONARY_FEATS}")

for feat in NON_STATIONARY_FEATS:
    diff_col = f"{feat}_diff"
    total_df[diff_col] = total_df[feat].diff()
    print(f"  추가: {diff_col}")

# 차분 컬럼 정상성 재확인
if NON_STATIONARY_FEATS:
    diff_cols = [f"{f}_diff" for f in NON_STATIONARY_FEATS]
    total_df_nodiff = total_df[diff_cols].dropna()
    stat_diff = stationarity_diagnostics(total_df_nodiff, diff_cols)
    print("\n차분 후 정상성 재확인:")
    print(stat_diff.to_string())
    assert all(stat_diff['Verdict'].isin(['Stationary', 'Conflicting'])),         "경고: 2차 차분이 필요한 피처 존재"

print("\n[주의] *_diff 컬럼은 STEP 2 딥러닝 입력 전처리용.")
print("[주의] HMM 라벨 생성에는 원본 피처를 그대로 사용한다.")


---
## 4. HMM Input Construction

### 4a. Non-overlapping 20-day windows

t 시점 기준 미래 20일(t+1~t+20)을 집약 → 334개 관측값
HMM 학습 전용. 딥러닝 입력(X)과 완전히 분리.


In [ ]:
HMM_FEATURES = ['log_return', 'log_vol_5d', 'log_vol_20d',
                'log_vol_60d', 'vol_ratio_5_20']

def build_future_window_features(df, features, window=20):
    """Non-overlapping 20일 미래 집약 피처 (HMM 전용)"""
    result = {}
    for f in features:
        shifts = pd.concat([df[f].shift(-k) for k in range(1, window + 1)], axis=1)
        if 'return' in f:
            result[f'{f}_fwd_sum']  = shifts.sum(axis=1)
        elif 'ratio' in f:
            result[f'{f}_fwd_max']  = shifts.max(axis=1)
        else:
            result[f'{f}_fwd_mean'] = shifts.mean(axis=1)

    fwd_df = pd.DataFrame(result, index=df.index)
    return fwd_df.dropna()


hmm_raw = build_future_window_features(total_df, HMM_FEATURES, FUTURE_WINDOW)
print(f"Non-overlapping HMM 입력: {hmm_raw.shape}")
hmm_raw.describe().round(5)


In [ ]:
# RobustScaler: HMM 라벨 생성기는 unsupervised → 전체 기간 fit 유지.
# [주의] STEP 2 딥러닝에서는 학습 구간 only fit 후 transform으로 사용할 것.
scaler_nonoverlap = RobustScaler()
X_nonoverlap = scaler_nonoverlap.fit_transform(hmm_raw.values)
X_nonoverlap_index = hmm_raw.index

print(f"Non-overlapping 입력 shape: {X_nonoverlap.shape}")
print(f"기간: {hmm_raw.index.min().date()} ~ {hmm_raw.index.max().date()}")


### 4b. Overlapping 20-day windows (v2)

매일 계산 → ~6500 샘플. sharpe/vol/drawdown/up_ratio 피처.

In [ ]:
def build_overlapping_features_v2(df, window=20):
    """
    Overlapping 윈도우 v2 피처:
    sharpe_fwd, vol_level_fwd, drawdown_fwd, up_ratio_fwd
    인접 윈도우 19일 겹침 → 자연스러운 스무딩
    """
    n = len(df)
    rows, idx = [], []
    close  = df['close'].values
    ret    = df['log_return'].values
    lvol5  = df['log_vol_5d'].values
    lvol20 = df['log_vol_20d'].values
    lvol60 = df['log_vol_60d'].values

    for i in range(n - window):
        future_ret   = ret[i+1: i+1+window]
        future_close = close[i+1: i+1+window]
        mu, sig = future_ret.mean(), future_ret.std()

        sharpe_fwd    = mu / sig if sig > 0 else 0
        vol_level_fwd = np.mean([lvol5[i+1:i+1+window].mean(),
                                  lvol20[i+1:i+1+window].mean(),
                                  lvol60[i+1:i+1+window].mean()])
        roll_max      = np.maximum.accumulate(future_close)
        drawdown_fwd  = ((future_close - roll_max) / roll_max).min()
        up_ratio_fwd  = (future_ret > 0).mean()

        rows.append({'sharpe_fwd': sharpe_fwd, 'vol_level_fwd': vol_level_fwd,
                     'drawdown_fwd': drawdown_fwd, 'up_ratio_fwd': up_ratio_fwd})
        idx.append(df.index[i])

    return pd.DataFrame(rows, index=idx)


hmm_overlap = build_overlapping_features_v2(total_df, window=FUTURE_WINDOW)
scaler_overlap = RobustScaler()
X_overlap = scaler_overlap.fit_transform(hmm_overlap.values)
X_overlap_index = hmm_overlap.index

print(f"Overlapping 입력 shape: {X_overlap.shape}")
print(f"기간: {hmm_overlap.index.min().date()} ~ {hmm_overlap.index.max().date()}")


---
## 5. Multi-Variant HMM Training

In [ ]:
def fit_hmm_multi_seed(model_factory, X, n_seeds=10, verbose=False):
    """
    여러 초기값(seed)으로 HMM 학습 → 수렴한 모델 중 최고 log-likelihood 반환.
    Baum-Welch EM의 로컬 최적해 함정 완화.
    """
    best_ll, best_model, n_converged = -np.inf, None, 0
    for seed in range(n_seeds):
        m = model_factory(seed)
        try:
            m.fit(X)
            if not m.monitor_.converged:
                continue
            n_converged += 1
            ll = m.score(X)
            if ll > best_ll:
                best_ll, best_model = ll, m
        except Exception:
            continue
    if verbose:
        print(f"  수렴률: {n_converged}/{n_seeds}")
    return best_model, best_ll, n_converged


def compute_bic(model, X, n_components, n_features, model_type='gaussian_full'):
    """변형별 파라미터 수 계산 후 BIC/AIC 반환"""
    n = len(X)
    if model_type == 'gaussian_full':
        n_params = (n_components*(n_components-1) + (n_components-1)
                    + n_components*n_features
                    + n_components*n_features*(n_features+1)//2)
    elif model_type == 'gaussian_diag':
        n_params = (n_components*(n_components-1) + (n_components-1)
                    + n_components*n_features*2)
    elif model_type == 'gmm':
        n_mix = 2
        n_params = (n_components*(n_components-1) + (n_components-1)
                    + n_components*(n_mix-1)
                    + n_components*n_mix*n_features*2)
    else:
        n_params = (n_components*(n_components-1) + (n_components-1)
                    + n_components*n_features
                    + n_components*n_features*(n_features+1)//2)

    ll = model.score(X) * n
    bic = -2*ll + n_params*np.log(n)
    aic = -2*ll + 2*n_params
    return ll/n, bic, aic, n_params


print("공통 헬퍼 함수 정의 완료")


### 5a. GaussianHMM — full vs diag covariance

In [ ]:
# ── GaussianHMM-full (기존 방식 강화) ──
# Non-overlapping X 기준 (334 샘플 × 5 피처)
results_full = []
models_full  = {}

print(f"{'N':>3} {'LogLL':>10} {'BIC':>12} {'AIC':>12} "
      f"{'MinRatio%':>10} {'MinPersist':>11} {'Conv':>6}")
print("-" * 70)

for n in STATE_RANGE:
    def factory_full(seed, _n=n):
        return hmm.GaussianHMM(n_components=_n, covariance_type='full',
                               n_iter=500, tol=1e-5, random_state=seed)

    m, ll, n_conv = fit_hmm_multi_seed(factory_full, X_nonoverlap, N_SEEDS)
    if m is None:
        print(f"{n:>3} FAILED")
        continue

    log_ll, bic, aic, _ = compute_bic(m, X_nonoverlap, n,
                                       X_nonoverlap.shape[1], 'gaussian_full')
    states    = m.predict(X_nonoverlap)
    min_ratio = pd.Series(states).value_counts(normalize=True).min() * 100
    min_p     = np.diag(m.transmat_).min()

    results_full.append({'variant': 'A_full', 'N': n, 'LogLL': log_ll,
                         'BIC': bic, 'AIC': aic, 'min_ratio': min_ratio,
                         'min_persist': min_p, 'conv_rate': n_conv})
    models_full[n] = (m, states, ll)
    print(f"{n:>3} {log_ll:>10.2f} {bic:>12.2f} {aic:>12.2f} "
          f"{min_ratio:>10.2f} {min_p:>11.4f} {n_conv:>4}/{N_SEEDS}")


In [ ]:
# ── GaussianHMM-diag (공분산 자유도 축소) ──
results_diag = []
models_diag  = {}

print(f"{'N':>3} {'LogLL':>10} {'BIC':>12} {'AIC':>12} "
      f"{'MinRatio%':>10} {'MinPersist':>11} {'Conv':>6}")
print("-" * 70)

for n in STATE_RANGE:
    def factory_diag(seed, _n=n):
        return hmm.GaussianHMM(n_components=_n, covariance_type='diag',
                               n_iter=500, tol=1e-5, random_state=seed)

    m, ll, n_conv = fit_hmm_multi_seed(factory_diag, X_nonoverlap, N_SEEDS)
    if m is None:
        print(f"{n:>3} FAILED")
        continue

    log_ll, bic, aic, _ = compute_bic(m, X_nonoverlap, n,
                                       X_nonoverlap.shape[1], 'gaussian_diag')
    states    = m.predict(X_nonoverlap)
    min_ratio = pd.Series(states).value_counts(normalize=True).min() * 100
    min_p     = np.diag(m.transmat_).min()

    results_diag.append({'variant': 'B_diag', 'N': n, 'LogLL': log_ll,
                         'BIC': bic, 'AIC': aic, 'min_ratio': min_ratio,
                         'min_persist': min_p, 'conv_rate': n_conv})
    models_diag[n] = (m, states, ll)
    print(f"{n:>3} {log_ll:>10.2f} {bic:>12.2f} {aic:>12.2f} "
          f"{min_ratio:>10.2f} {min_p:>11.4f} {n_conv:>4}/{N_SEEDS}")


### 5b. GMM-HMM (n_mix=2, diag)

Student-t HMM 대체: `hmmlearn`이 t-emission을 기본 제공하지 않으며,
외부 라이브러리(pyhmmix 등)는 Colab 의존성 충돌 가능성이 높다.
GMM-HMM(n_mix=2)은 Gaussian 혼합으로 fat-tail을 근사할 수 있어 실용적 대체로 충분하다
(Bishop 2006).


In [ ]:
# ── GMMHMM (각 레짐을 2개 가우시안 혼합 → 첨도/fat-tail 표현) ──
results_gmm = []
models_gmm  = {}

print(f"{'N':>3} {'LogLL':>10} {'BIC':>12} {'AIC':>12} "
      f"{'MinRatio%':>10} {'MinPersist':>11} {'Conv':>6}")
print("-" * 70)

for n in STATE_RANGE:
    def factory_gmm(seed, _n=n):
        return hmm.GMMHMM(n_components=_n, n_mix=2, covariance_type='diag',
                          n_iter=300, tol=1e-4, random_state=seed)

    m, ll, n_conv = fit_hmm_multi_seed(factory_gmm, X_nonoverlap, N_SEEDS)
    if m is None:
        # n_mix=2 실패 시 n_mix=1(=GaussianHMM-diag) 폴백
        def factory_gmm_fallback(seed, _n=n):
            return hmm.GMMHMM(n_components=_n, n_mix=1, covariance_type='diag',
                              n_iter=300, tol=1e-4, random_state=seed)
        m, ll, n_conv = fit_hmm_multi_seed(factory_gmm_fallback, X_nonoverlap, N_SEEDS)
        print(f"  N={n}: n_mix=2 실패 → n_mix=1 폴백")
        if m is None:
            print(f"{n:>3} FAILED")
            continue

    log_ll, bic, aic, _ = compute_bic(m, X_nonoverlap, n,
                                       X_nonoverlap.shape[1], 'gmm')
    states    = m.predict(X_nonoverlap)
    min_ratio = pd.Series(states).value_counts(normalize=True).min() * 100
    min_p     = np.diag(m.transmat_).min()

    results_gmm.append({'variant': 'C_gmm', 'N': n, 'LogLL': log_ll,
                        'BIC': bic, 'AIC': aic, 'min_ratio': min_ratio,
                        'min_persist': min_p, 'conv_rate': n_conv})
    models_gmm[n] = (m, states, ll)
    print(f"{n:>3} {log_ll:>10.2f} {bic:>12.2f} {aic:>12.2f} "
          f"{min_ratio:>10.2f} {min_p:>11.4f} {n_conv:>4}/{N_SEEDS}")


### 5c. Overlapping 피처 GaussianHMM (v2 피처)

In [ ]:
# Overlapping 경로도 multi-seed로 강화
results_overlap = []
models_overlap  = {}

print(f"{'N':>3} {'LogLL':>10} {'BIC':>12} {'AIC':>12} "
      f"{'MinRatio%':>10} {'MinPersist':>11} {'Conv':>6}")
print("-" * 70)

for n in STATE_RANGE:
    def factory_overlap(seed, _n=n):
        return hmm.GaussianHMM(n_components=_n, covariance_type='full',
                               n_iter=300, tol=1e-4, random_state=seed)

    m, ll, n_conv = fit_hmm_multi_seed(factory_overlap, X_overlap, N_SEEDS)
    if m is None:
        print(f"{n:>3} FAILED")
        continue

    log_ll, bic, aic, _ = compute_bic(m, X_overlap, n,
                                       X_overlap.shape[1], 'gaussian_full')
    states    = m.predict(X_overlap)
    min_ratio = pd.Series(states).value_counts(normalize=True).min() * 100
    min_p     = np.diag(m.transmat_).min()

    results_overlap.append({'variant': 'D_overlap', 'N': n, 'LogLL': log_ll,
                            'BIC': bic, 'AIC': aic, 'min_ratio': min_ratio,
                            'min_persist': min_p, 'conv_rate': n_conv})
    models_overlap[n] = (m, states, ll)
    print(f"{n:>3} {log_ll:>10.2f} {bic:>12.2f} {aic:>12.2f} "
          f"{min_ratio:>10.2f} {min_p:>11.4f} {n_conv:>4}/{N_SEEDS}")


### 5d. HSMM (Negative-Binomial duration)

In [ ]:
# HSMM: hsmmlearn 성공 시 학습, 실패 시 duration 분포 분석으로 폴백
hsmm_result = None

if HSMM_AVAILABLE:
    try:
        # hsmmlearn API: GaussianHSMM
        from hsmmlearn.hsmm import GaussianHSMM
        N_HSMM = 5
        hsmm_model = GaussianHSMM(n_states=N_HSMM, n_durations=30,
                                   random_state=RANDOM_STATE)
        hsmm_model.fit(X_nonoverlap)
        hsmm_states = hsmm_model.decode(X_nonoverlap)
        hsmm_result = {'model': hsmm_model, 'states': hsmm_states, 'N': N_HSMM}
        print(f"HSMM 학습 성공 (N={N_HSMM})")
    except Exception as e:
        print(f"HSMM 학습 실패: {e}")
        HSMM_AVAILABLE = False

if not HSMM_AVAILABLE:
    print("HSMM 폴백: GaussianHMM-full N=5 기준 duration 분포 분석")
    if 5 in models_full:
        m_ref, states_ref, _ = models_full[5]
        durations = []
        i = 0
        while i < len(states_ref):
            j = i
            while j < len(states_ref) and states_ref[j] == states_ref[i]:
                j += 1
            durations.append(j - i)
            i = j
        dur_s = pd.Series(durations)
        print(f"지속기간 통계: 평균={dur_s.mean():.1f}일 | 중앙={dur_s.median():.1f}일 "
              f"| 1일전환={( dur_s==1).sum()}회 ({(dur_s==1).mean()*100:.1f}%)")

        # Negative-Binomial 적합 (duration 분포 진단용)
        from scipy.stats import nbinom
        dur_arr = dur_s.values
        mean_d, var_d = dur_arr.mean(), dur_arr.var()
        if var_d > mean_d:
            p_nb = mean_d / var_d
            r_nb = mean_d**2 / (var_d - mean_d)
            print(f"Negative-Binomial 적합: r={r_nb:.2f}, p={p_nb:.3f}")
        else:
            print("분산 <= 평균 → 포아송 또는 기하 분포 적합 권장")

        fig, ax = plt.subplots(figsize=(9, 3))
        ax.hist(dur_arr, bins=40, edgecolor='black', alpha=0.7, density=True)
        ax.set(title='레짐 지속기간 분포 (HSMM 폴백 진단)', xlabel='일수')
        plt.tight_layout(); plt.show()


---
## 6. Model Selection — 통합 기준표

In [ ]:
def event_match_score_prelim(states, index, down_regimes, n_components):
    """
    임시 이벤트 F1 계산 (포스터리어 없는 hard label 기준).
    down_regimes: 레짐 정렬 전 원시 상태에서의 하위 절반 집합 (수익률 기준).
    """
    EVENTS = {
        '닷컴버블': ('2000-03-01', '2002-10-09'),
        '금융위기': ('2007-10-01', '2009-03-09'),
        '코로나':   ('2020-02-19', '2020-04-30'),
        '긴축':     ('2022-01-01', '2022-12-31'),
    }
    regime_series = pd.Series(states, index=index)
    y_true_all, y_pred_all = [], []

    for s, e in EVENTS.values():
        mask = (regime_series.index >= s) & (regime_series.index <= e)
        if mask.sum() == 0:
            continue
        sub_dates = regime_series.index[mask]
        y_true = pd.Series(1, index=sub_dates)
        y_pred_event = regime_series[mask].isin(down_regimes).astype(int)
        full_range = regime_series.index
        yt = pd.Series(0, index=full_range)
        yt.loc[sub_dates] = 1
        yp = regime_series.isin(down_regimes).astype(int)
        y_true_all.append(yt.values)
        y_pred_all.append(yp.values)

    if not y_true_all:
        return 0.0

    yt_concat = np.concatenate(y_true_all)
    yp_concat = np.concatenate(y_pred_all)
    return f1_score(yt_concat, yp_concat, zero_division=0)


# 통합 결과 테이블 구성
all_results = results_full + results_diag + results_gmm + results_overlap
summary_df  = pd.DataFrame(all_results)

# 임시 event_F1 계산 (정렬 전 raw states)
def get_down_regimes(states, n):
    """수익률 기준 하위 절반 레짐 식별"""
    returns_ref = total_df.loc[X_nonoverlap_index, 'log_return']
    if len(returns_ref) != len(states):
        return set(range(n // 2))
    regime_ret = {s: returns_ref[np.array(states) == s].mean()
                  for s in range(n)}
    sorted_r = sorted(regime_ret, key=lambda x: regime_ret[x])
    return set(sorted_r[:n // 2])


f1_scores = []
for _, row in summary_df.iterrows():
    v, n = row['variant'], int(row['N'])
    try:
        if v == 'A_full' and n in models_full:
            m, st, _ = models_full[n]
            down_r = get_down_regimes(st, n)
            f1 = event_match_score_prelim(st, X_nonoverlap_index, down_r, n)
        elif v == 'B_diag' and n in models_diag:
            m, st, _ = models_diag[n]
            down_r = get_down_regimes(st, n)
            f1 = event_match_score_prelim(st, X_nonoverlap_index, down_r, n)
        elif v == 'C_gmm' and n in models_gmm:
            m, st, _ = models_gmm[n]
            down_r = get_down_regimes(st, n)
            f1 = event_match_score_prelim(st, X_nonoverlap_index, down_r, n)
        elif v == 'D_overlap' and n in models_overlap:
            m, st, _ = models_overlap[n]
            returns_ov = total_df.loc[X_overlap_index, 'log_return']
            regime_ret = {s: returns_ov[np.array(st) == s].mean() for s in range(n)}
            sorted_r   = sorted(regime_ret, key=lambda x: regime_ret[x])
            down_r     = set(sorted_r[:n // 2])
            f1 = event_match_score_prelim(st, X_overlap_index, down_r, n)
        else:
            f1 = np.nan
    except Exception:
        f1 = np.nan
    f1_scores.append(round(f1, 4))

summary_df['event_F1'] = f1_scores
summary_df = summary_df.sort_values(['variant', 'N'])

print("=" * 90)
print("통합 모델 선택 기준표")
print("=" * 90)
cols_show = ['variant', 'N', 'LogLL', 'BIC', 'AIC', 'min_ratio', 'min_persist', 'conv_rate', 'event_F1']
print(summary_df[cols_show].to_string(index=False))


In [ ]:
# ── 모델 선택 기준 (가중치 명시) ──
# 1. 수렴률 >= 7/10
# 2. min_persist >= 0.90
# 3. min_ratio >= 5%
# 4. BIC 최소 또는 BIC 차이 <= 2 (Kass-Raftery)
# 5. event_F1 최대

eligible = summary_df[
    (summary_df['conv_rate'] >= 7) &
    (summary_df['min_persist'] >= 0.90) &
    (summary_df['min_ratio'] >= 5.0)
].copy()

if len(eligible) == 0:
    print("기준 완화: min_persist >= 0.85")
    eligible = summary_df[
        (summary_df['conv_rate'] >= 5) &
        (summary_df['min_persist'] >= 0.85) &
        (summary_df['min_ratio'] >= 3.0)
    ].copy()

# BIC 정규화 후 composite score = 0.5*(1-BIC_norm) + 0.5*event_F1
eligible['BIC_norm'] = (eligible['BIC'] - eligible['BIC'].min()) /                         (eligible['BIC'].max() - eligible['BIC'].min() + 1e-9)
eligible['score'] = 0.5 * (1 - eligible['BIC_norm']) + 0.5 * eligible['event_F1'].fillna(0)

best_row    = eligible.loc[eligible['score'].idxmax()]
BEST_VARIANT = best_row['variant']
BEST_N       = int(best_row['N'])

print(f"\n최종 선택: variant={BEST_VARIANT}, N={BEST_N}")
print(f"  BIC={best_row['BIC']:.2f} | min_persist={best_row['min_persist']:.4f} "
      f"| event_F1={best_row['event_F1']:.4f}")

# 선택된 모델 및 입력 데이터
variant_map = {
    'A_full':    (models_full,    X_nonoverlap, X_nonoverlap_index, 'gaussian_full', scaler_nonoverlap),
    'B_diag':    (models_diag,    X_nonoverlap, X_nonoverlap_index, 'gaussian_diag', scaler_nonoverlap),
    'C_gmm':     (models_gmm,     X_nonoverlap, X_nonoverlap_index, 'gmm',           scaler_nonoverlap),
    'D_overlap': (models_overlap, X_overlap,    X_overlap_index,    'gaussian_full', scaler_overlap),
}
_models_dict, X_best, idx_best, _vtype, scaler_best = variant_map[BEST_VARIANT]
best_model, best_states_raw, _ = _models_dict[BEST_N]
print(f"\n수렴 여부: {best_model.monitor_.converged}")
print(f"최종 LogLL: {best_model.score(X_best):.4f}")


---
## 7. Posterior Computation & Soft Labels

In [ ]:
def build_soft_label_frame(model, X, window_index, daily_index, n_components, sorted_regimes):
    """
    posteriors (T_windows × N) → 일별 daily_index로 broadcast.
    Non-overlapping: 각 윈도우 시작점에서 20일 동일값 broadcast.
    Overlapping: 이미 일별이므로 직접 정렬.
    """
    posteriors_raw = model.predict_proba(X)            # (T, N)
    posteriors     = posteriors_raw[:, sorted_regimes] # 수익률 오름차순 정렬

    # window_index가 daily_index의 90% 이상이면 일별 데이터(overlapping 경로)
    is_overlapping = (len(window_index) >= len(daily_index) * 0.90)

    if is_overlapping:
        post_df = pd.DataFrame(
            posteriors,
            index=window_index,
            columns=[f'regime_prob_{i}' for i in range(n_components)]
        )
        return post_df.reindex(daily_index).ffill()
    else:
        # Non-overlapping: 윈도우 시작 → 20일 broadcast
        post_series = {f'regime_prob_{i}': {} for i in range(n_components)}
        all_dates   = pd.bdate_range(daily_index.min(), daily_index.max())

        for wi, wdate in enumerate(window_index):
            wdate_pos = all_dates.get_loc(wdate) if wdate in all_dates else None
            if wdate_pos is None:
                continue
            end_pos = min(wdate_pos + 20, len(all_dates))
            for d in all_dates[wdate_pos:end_pos]:
                for i in range(n_components):
                    post_series[f'regime_prob_{i}'][d] = posteriors[wi, i]

        post_df = pd.DataFrame(post_series)
        return post_df.reindex(daily_index).ffill()


# 레짐 재정렬: 수익률 오름차순
returns_ref = total_df.loc[idx_best, 'log_return']
regime_ret  = {s: returns_ref[best_states_raw == s].mean() for s in range(BEST_N)}
sorted_regimes = sorted(regime_ret, key=lambda x: regime_ret[x])
remap          = {old: new for new, old in enumerate(sorted_regimes)}
best_states    = np.array([remap[s] for s in best_states_raw])

# Posterior 계산
posteriors_sorted = best_model.predict_proba(X_best)[:, sorted_regimes]
assert np.allclose(posteriors_sorted.sum(axis=1), 1.0), "posterior 합 != 1"
print(f"Posterior shape: {posteriors_sorted.shape}")
print(f"합 검증 통과: {np.allclose(posteriors_sorted.sum(axis=1), 1.0)}")

# 일별 soft label 프레임
post_daily = build_soft_label_frame(
    best_model, X_best, idx_best,
    total_df.index, BEST_N, sorted_regimes
)
print(f"일별 Soft Label shape: {post_daily.shape}")
print(post_daily.head())

# 평균 엔트로피 (값이 높으면 레짐 경계가 불명확 → STEP 2에서 soft label 활용 권장)
entropy = -np.sum(posteriors_sorted * np.log(posteriors_sorted + 1e-9), axis=1)
print(f"\n평균 posterior 엔트로피: {entropy.mean():.4f} (max={entropy.max():.4f})")


---
## 8. Regime Labeling, Sorting, Validation

In [ ]:
# 수익률 오름차순 정렬 결과 확인
regime_series = pd.Series(best_states, index=idx_best, name='regime_label')
returns_all   = total_df.loc[idx_best, 'log_return']

print("레짐별 통계 (정렬 후):")
print(f"{'레짐':>4} {'일수':>6} {'비율%':>7} {'μ(%)':>9} {'σ(%)':>9} {'샤프':>7}")
print("-" * 45)
for s in range(BEST_N):
    mask = regime_series == s
    r    = returns_all[mask] * 100
    sp   = r.mean() / r.std() * np.sqrt(252) if r.std() > 0 else 0
    print(f"{s:>4} {mask.sum():>6} {mask.mean()*100:>7.1f} "
          f"{r.mean():>9.4f} {r.std():>9.4f} {sp:>7.3f}")

# 단조증가 검증 (수익률 기준 정렬 확인)
means_sorted = [returns_all[regime_series == s].mean() for s in range(BEST_N)]
assert all(means_sorted[i] <= means_sorted[i+1] for i in range(len(means_sorted)-1)),     "레짐 수익률이 단조증가하지 않음 — 정렬 로직 확인"
print("\n수익률 단조증가 검증 통과")


In [ ]:
# 시각화: S&P500 + 레짐 배경
REGIME_COLORS_SORTED = {
    0: '#8B0000', 1: '#FF6B6B', 2: '#AAAAAA',
    3: '#90EE90', 4: '#006400', 5: '#003300'
}
close_ref = total_df.loc[idx_best, 'close']

fig, axes = plt.subplots(3, 1, figsize=(18, 12),
                          gridspec_kw={'height_ratios': [4, 1.5, 1.5]})
ax1 = axes[0]
ax1.plot(close_ref.index, close_ref.values, color='black', lw=0.9, zorder=3)
prev, si = None, 0
for i in range(len(regime_series)):
    r = regime_series.iloc[i]
    if prev is None:
        prev, si = r, i
    elif r != prev:
        ax1.axvspan(regime_series.index[si], regime_series.index[i],
                    color=REGIME_COLORS_SORTED.get(prev, '#CCCCCC'), alpha=0.25)
        prev, si = r, i
ax1.axvspan(regime_series.index[si], regime_series.index[-1],
            color=REGIME_COLORS_SORTED.get(prev, '#CCCCCC'), alpha=0.25)
patches = [mpatches.Patch(color=REGIME_COLORS_SORTED.get(r, '#CCCCCC'), alpha=0.6, label=f"Regime {r}")
           for r in range(BEST_N)]
ax1.legend(handles=patches, loc='upper left', fontsize=9)
ax1.set(title=f"S&P500 + HMM Regime ({BEST_VARIANT}, N={BEST_N})", ylabel='Price')
ax1.grid(True, alpha=0.3)

axes[1].bar(regime_series.index, regime_series.values,
            color=[REGIME_COLORS_SORTED.get(r, '#CCCCCC') for r in regime_series], width=1)
axes[1].set(title='Regime State', ylabel='Regime', yticks=range(BEST_N))
axes[1].grid(True, alpha=0.3)

axes[2].bar(returns_all.index, returns_all.values * 100,
            color=[REGIME_COLORS_SORTED.get(r, '#CCCCCC') for r in regime_series], width=1)
axes[2].axhline(0, color='black', lw=0.8)
axes[2].set(title='Log Returns by Regime (%)', ylabel='Return (%)')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('regime_final.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# 전이 행렬 분석
trans_sorted = best_model.transmat_[np.ix_(sorted_regimes, sorted_regimes)]
trans_df = pd.DataFrame(
    trans_sorted,
    index   = [f"From R{s}" for s in range(BEST_N)],
    columns = [f"To R{s}"   for s in range(BEST_N)]
)
print("전이 행렬:"); print(trans_df.round(4))

print("\n자기지속 확률 및 평균 지속 기간:")
for s in range(BEST_N):
    p = trans_sorted[s, s]
    avg_d = 1 / (1 - p) if p < 1 else np.inf
    print(f"  레짐 {s}: 지속확률 {p*100:.1f}% → 평균 {avg_d:.1f}일")

fig, ax = plt.subplots(figsize=(6, 4))
sns.heatmap(trans_df, annot=True, fmt='.3f', cmap='Blues',
            vmin=0, vmax=1, ax=ax, cbar_kws={'label': 'Probability'})
ax.set_title('HMM Transition Matrix (정렬 후)')
plt.tight_layout(); plt.show()


---
## 9. Quantitative Event-Matching Score

하위 절반 레짐({0, 1, ...})을 '하락 예측'으로 정의하여 F1/AUROC 계산.

In [ ]:
def event_match_score(regime_series, posteriors_df, events, down_regimes, n_components):
    """
    4대 경제 이벤트 구간에서 F1 / precision / recall / AUROC 계산.
    regime_series: pd.Series (날짜 인덱스, 정수 레짐값)
    posteriors_df: pd.DataFrame (날짜 인덱스, regime_prob_0..N-1)
    events: dict {이름: (시작, 끝)}
    down_regimes: set, 하락 레짐 번호
    """
    full_index = regime_series.index
    y_true_all = np.zeros(len(full_index), dtype=int)
    for s, e in events.values():
        mask = (full_index >= s) & (full_index <= e)
        y_true_all[mask] = 1

    y_pred_hard = regime_series.isin(down_regimes).astype(int).values

    # Soft score: 하락 레짐 posterior 합
    down_cols = [f'regime_prob_{r}' for r in sorted(down_regimes) if f'regime_prob_{r}' in posteriors_df.columns]
    if down_cols and len(posteriors_df) == len(full_index):
        post_aligned = posteriors_df.reindex(full_index).ffill().fillna(0)
        y_score = post_aligned[down_cols].sum(axis=1).values
    else:
        y_score = y_pred_hard.astype(float)

    results = {}
    for name, (s, e) in events.items():
        mask = (full_index >= s) & (full_index <= e)
        if mask.sum() == 0:
            continue
        yt_ev = y_true_all.copy()
        yp_ev = y_pred_hard.copy()
        ys_ev = y_score.copy()

        f1   = f1_score(yt_ev, yp_ev, zero_division=0)
        prec = precision_score(yt_ev, yp_ev, zero_division=0)
        rec  = recall_score(yt_ev, yp_ev, zero_division=0)
        try:
            auc = roc_auc_score(yt_ev, ys_ev)
        except Exception:
            auc = np.nan

        # 이벤트 구간 내 하락 레짐 점유율
        in_event_share = regime_series[mask].isin(down_regimes).mean() * 100
        results[name] = {'F1': round(f1, 4), 'Precision': round(prec, 4),
                         'Recall': round(rec, 4), 'AUROC': round(auc, 4),
                         'InEvent_DownShare%': round(in_event_share, 1)}

    return pd.DataFrame(results).T


EVENTS_EVAL = {
    '닷컴버블 붕괴':     ('2000-03-01', '2002-10-09'),
    '글로벌 금융위기':   ('2007-10-01', '2009-03-09'),
    '코로나19 쇼크':    ('2020-02-19', '2020-04-30'),
    '인플레 긴축사이클': ('2022-01-01', '2022-12-31'),
}

down_regimes_final = set(range(BEST_N // 2))  # 하위 절반
post_aligned = post_daily.reindex(regime_series.index).ffill().fillna(1 / BEST_N)

event_scores = event_match_score(
    regime_series, post_aligned, EVENTS_EVAL, down_regimes_final, BEST_N
)
print("정량 이벤트 매칭 점수:")
print(event_scores.to_string())
print(f"\n평균 F1: {event_scores['F1'].mean():.4f}")
print(f"평균 AUROC: {event_scores['AUROC'].mean():.4f}")


In [ ]:
# --- 9-B. Regime Backtesting (Oracle Upper Bound) ----------------------------
# regime_label[t] = future 20d regime known in advance (oracle upper bound)
# Sets realistic performance ceiling for the DL model in STEP 2

def compute_mdd(cum_series):
    return (cum_series / cum_series.cummax() - 1).min()

def annualized_sharpe(daily_rets, periods=252):
    mu, sig = daily_rets.mean(), daily_rets.std()
    return mu / sig * np.sqrt(periods) if sig > 0 else 0.0

# 1. Regime-conditional 20d forward return distribution
print("=" * 62)
print("Regime-conditional 20d fwd return  (regime_label[t] -> fwd 20d)")
print("=" * 62)

df_bt = dl_dataset[["regime_label", "close"]].copy()
df_bt["fwd_20d"] = df_bt["close"].shift(-20) / df_bt["close"] - 1

_rnames = {0: "Bear", 1: "Weak Bear", 2: "Neutral", 3: "Weak Bull", 4: "Bull"}
bt_stats = {}
for r in range(BEST_N):
    rets = df_bt.loc[df_bt["regime_label"] == r, "fwd_20d"].dropna()
    if len(rets) < 5:
        continue
    ann  = rets.mean() * (252 / 20)
    vol  = rets.std()  * np.sqrt(252 / 20)
    sr   = ann / vol if vol > 0 else 0.0
    hit  = (rets > 0).mean()
    name = _rnames.get(r, f"R{r}")
    bt_stats[r] = dict(name=name, mean=rets.mean(), ann=ann, vol=vol, sharpe=sr, hit=hit, n=len(rets))
    print(f"  R{r} {name:>10}: mean={rets.mean():>7.2%}  ann={ann:>7.1%}"
          f"  vol={vol:>7.1%}  Sharpe={sr:>5.2f}  hit={hit:>5.1%}  N={len(rets)}")

# 2. Oracle strategy: upper half regimes -> Long, lower half -> Cash
_bull_thr = int(np.ceil(BEST_N / 2))
print(f"\n[Oracle] R>={_bull_thr} -> Long  |  R<{_bull_thr} -> Cash")

df_bt["signal"]    = (df_bt["regime_label"] >= _bull_thr).astype(int)
df_bt["daily_ret"] = df_bt["close"].pct_change()
df_bt["strat_ret"] = df_bt["signal"].shift(1) * df_bt["daily_ret"]

_v      = df_bt[["strat_ret", "daily_ret"]].dropna()
_cum_s  = (1 + _v["strat_ret"]).cumprod()
_cum_bh = (1 + _v["daily_ret"]).cumprod()
_ss = annualized_sharpe(_v["strat_ret"])
_bs = annualized_sharpe(_v["daily_ret"])
_sa = _v["strat_ret"].mean() * 252
_ba = _v["daily_ret"].mean() * 252
_sm = compute_mdd(_cum_s)
_bm = compute_mdd(_cum_bh)

print(f"\n{'Metric':>14} {'Oracle':>12} {'Buy&Hold':>12}")
print("-" * 40)
print(f"{'Ann.Return':>14} {_sa:>11.1%} {_ba:>11.1%}")
print(f"{'Sharpe':>14} {_ss:>11.2f} {_bs:>11.2f}")
print(f"{'MDD':>14} {_sm:>11.1%} {_bm:>11.1%}")
print(f"{'Cum.Return':>14} {_cum_s.iloc[-1]-1:>11.1%} {_cum_bh.iloc[-1]-1:>11.1%}")
print("[Note] Oracle = upper bound; actual DL performance will be lower")

# 3. Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
_cum_s.plot(ax=axes[0], label=f"Oracle (Long R{_bull_thr}+)", color="steelblue", lw=1.5)
_cum_bh.plot(ax=axes[0], label="Buy & Hold", color="gray", ls="--", alpha=0.8)
axes[0].set_title("Cumulative Return: Oracle vs Buy&Hold")
axes[0].set_ylabel("Cumulative Return (x)")
axes[0].legend(); axes[0].grid(True, alpha=0.3)

_dd_s  = _cum_s  / _cum_s.cummax()  - 1
_dd_bh = _cum_bh / _cum_bh.cummax() - 1
axes[1].fill_between(_dd_s.index,  _dd_s.values,  alpha=0.5, color="steelblue",
                     label=f"Oracle  MDD={_sm:.1%}")
axes[1].fill_between(_dd_bh.index, _dd_bh.values, alpha=0.3, color="gray",
                     label=f"B&H  MDD={_bm:.1%}")
axes[1].set_title("Drawdown Comparison")
axes[1].set_ylabel("Drawdown")
axes[1].legend(); axes[1].grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

backtest_results = dict(
    strat_sharpe=_ss, bnh_sharpe=_bs,
    strat_ann_ret=_sa, bnh_ann_ret=_ba,
    strat_mdd=_sm, bnh_mdd=_bm
)

---
## 10. Output Dataset Build

`sp500_regime_dataset_final.csv` + `regime_metadata.json`

In [ ]:
# Track1/Track2 피처 + 비정상 차분 컬럼 + regime_label + regime_prob_* + close
TRACK2_DIFF_FEATS = [f"{f}_diff" for f in NON_STATIONARY_FEATS
                     if f"{f}_diff" in total_df.columns]

# 공통 날짜 인덱스
labeled_idx = regime_series.index.intersection(total_df.index)
dl_dataset  = pd.DataFrame(index=labeled_idx)

for f in TRACK1_FEATS + TRACK2_FEATS + TRACK2_DIFF_FEATS:
    if f in total_df.columns:
        dl_dataset[f] = total_df.loc[labeled_idx, f]

dl_dataset['regime_label'] = regime_series.loc[labeled_idx].astype(int)

# Soft posterior 컬럼 추가
prob_cols = [f'regime_prob_{i}' for i in range(BEST_N)]
post_for_save = post_daily.reindex(labeled_idx).ffill().fillna(1 / BEST_N)
for col in prob_cols:
    dl_dataset[col] = post_for_save[col] if col in post_for_save.columns else 1 / BEST_N

dl_dataset['close'] = total_df.loc[labeled_idx, 'close']
dl_dataset = dl_dataset.dropna(subset=TRACK1_FEATS + TRACK2_FEATS)

print(f"딥러닝 데이터셋: {dl_dataset.shape}")
print(f"기간: {dl_dataset.index.min().date()} ~ {dl_dataset.index.max().date()}")
print(f"Track1: {len(TRACK1_FEATS)}개 | Track2: {len(TRACK2_FEATS)}개 | "
      f"diff: {len(TRACK2_DIFF_FEATS)}개 | soft_prob: {BEST_N}개")
print(f"\nTarget 분포:")
for s in range(BEST_N):
    cnt = (dl_dataset['regime_label'] == s).sum()
    print(f"  레짐 {s}: {cnt:>5}일 ({cnt/len(dl_dataset)*100:.1f}%)")


In [ ]:
# posterior 합 = 1 검증
prob_sum = dl_dataset[prob_cols].sum(axis=1)
assert np.allclose(prob_sum, 1.0, atol=1e-3), f"posterior 합 이상: {prob_sum.describe()}"
print(f"posterior 합 검증 통과: min={prob_sum.min():.6f}, max={prob_sum.max():.6f}")

# CSV 저장
OUTPUT_CSV = 'sp500_regime_dataset_final.csv'
dl_dataset.to_csv(OUTPUT_CSV)
print(f"\n저장 완료: {OUTPUT_CSV}")
print(f"컬럼 목록: {list(dl_dataset.columns)}")


In [ ]:
# regime_metadata.json 저장
import os

# RobustScaler 파라미터
scaler_params = {
    'center': scaler_best.center_.tolist(),
    'scale':  scaler_best.scale_.tolist(),
}

# 전이 행렬
trans_matrix = trans_sorted.tolist()

# 자기지속 확률
self_persist = {str(s): float(trans_sorted[s, s]) for s in range(BEST_N)}

# 레짐 평균 수익률
regime_means = {str(s): float(returns_ref[best_states == s].mean())
                for s in range(BEST_N) if (best_states == s).sum() > 0}

# 정상성 보고서
stat_report = stat_df[['ADF_p', 'KPSS_p', 'Verdict']].to_dict(orient='index')

metadata = {
    'best_variant':         BEST_VARIANT,
    'best_n':               BEST_N,
    'random_state':         RANDOM_STATE,
    'n_seeds':              N_SEEDS,
    'future_window':        FUTURE_WINDOW,
    'start_date':           START_DATE,
    'end_date':             END_DATE,
    'regime_means':         regime_means,
    'transition_matrix':    trans_matrix,
    'self_persistence':     self_persist,
    'event_f1_score':       float(event_scores['F1'].mean()),
    'event_auroc_score':    float(event_scores['AUROC'].mean()),
    'event_scores_detail':  event_scores.to_dict(orient='index'),
    'convergence_history':  list(best_model.monitor_.history),
    'stationarity_report':  stat_report,
    'scaler_params':        scaler_params,
    'posterior_entropy_mean': float(entropy.mean()),
    'note_leakage':         '이 scaler/HMM은 전체 기간 fit(라벨 생성기 unsupervised). STEP2 DL에서는 학습셋 only fit 사용.',
}

OUTPUT_META = os.path.join('..', 'output', 'regime_metadata.json')
os.makedirs(os.path.dirname(OUTPUT_META), exist_ok=True)
with open(OUTPUT_META, 'w', encoding='utf-8') as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

print(f"저장 완료: {OUTPUT_META}")
print(json.dumps({k: metadata[k] for k in
                  ['best_variant', 'best_n', 'event_f1_score', 'event_auroc_score']},
                 ensure_ascii=False, indent=2))


In [ ]:
def run_final_checks():
    """10개 최종 검증 — 모두 PASS 출력 시 개편 완료"""
    results = []

    def chk(name, cond, msg=''):
        status = 'PASS' if cond else 'FAIL'
        results.append((status, name, msg))
        print(f"  [{status}] {name}" + (f" — {msg}" if msg else ''))

    print("=" * 60)
    print("최종 검증 (run_final_checks)")
    print("=" * 60)

    # 1. 데이터 무결성 (total_df *_diff 첫행 NaN은 정상 — dl_dataset Track1/2만 검사)
    _key_cols = TRACK1_FEATS + TRACK2_FEATS
    _nan_key  = dl_dataset[_key_cols].isna().sum().sum()
    chk("1. 데이터 무결성",
        _nan_key == 0 and len(dl_dataset) >= 5000,
        f"rows={len(dl_dataset)}, NaN(Track1+2)={_nan_key}")

    # 2. 정상성 셀 — *_diff 컬럼 생성
    chk("2. 비정상 피처 차분 컬럼",
        all(f"{f}_diff" in total_df.columns for f in NON_STATIONARY_FEATS),
        f"NON_STATIONARY: {NON_STATIONARY_FEATS}")

    # 3. 수렴률 — 어느 (variant, N) 에서든 >= 7/10 존재
    max_conv = summary_df['conv_rate'].max()
    chk("3. 최대 수렴률 >= 7",
        max_conv >= 7, f"max_conv_rate={max_conv}")

    # 4. 모델 선택 표 출력 + BEST_VARIANT, BEST_N 명시
    chk("4. BEST_VARIANT/N 결정",
        BEST_VARIANT != '' and BEST_N > 0,
        f"BEST_VARIANT={BEST_VARIANT}, BEST_N={BEST_N}")

    # 5. Posterior 합 = 1
    prob_cols_chk = [f'regime_prob_{i}' for i in range(BEST_N)]
    prob_sum_chk  = dl_dataset[prob_cols_chk].sum(axis=1)
    chk("5. Posterior 합 = 1",
        np.allclose(prob_sum_chk, 1.0, atol=1e-3),
        f"min={prob_sum_chk.min():.6f}, max={prob_sum_chk.max():.6f}")

    # 6. 레짐 수익률 단조증가
    m_sorted = [dl_dataset[dl_dataset['regime_label'] == s]['log_return'].mean()
                for s in range(BEST_N)]
    chk("6. 레짐 수익률 단조증가",
        all(m_sorted[i] <= m_sorted[i+1] for i in range(len(m_sorted)-1)),
        str([round(x, 5) for x in m_sorted]))

    # 7. CSV 컬럼 갯수
    expected_min = len(TRACK1_FEATS) + len(TRACK2_FEATS) + 1 + BEST_N + 1
    chk("7. CSV 컬럼 갯수",
        len(dl_dataset.columns) >= expected_min,
        f"actual={len(dl_dataset.columns)}, expected>={expected_min}")

    # 8. 이벤트 F1 확인 (>= 0)
    avg_f1 = event_scores['F1'].mean()
    chk("8. 이벤트 평균 F1 >= 0",
        avg_f1 >= 0,
        f"avg_F1={avg_f1:.4f}")

    # 9. STEP 2 호환성 smoke test
    try:
        df_reload = pd.read_csv(OUTPUT_CSV, parse_dates=[0], index_col=0)
        ok = (df_reload['regime_label'].dtype in [int, 'int64', 'int32']
              and df_reload[f'regime_prob_0'].dtype == float)
        chk("9. STEP2 호환 smoke test", ok,
            f"regime_label dtype={df_reload['regime_label'].dtype}")
    except Exception as e:
        chk("9. STEP2 호환 smoke test", False, str(e))

    # 10. metadata.json 키 확인
    try:
        with open(OUTPUT_META, 'r', encoding='utf-8') as f:
            meta = json.load(f)
        ok = ('best_variant' in meta and 'event_f1_score' in meta)
        chk("10. metadata.json 키 확인", ok)
    except Exception as e:
        chk("10. metadata.json 키 확인", False, str(e))

    print("=" * 60)
    n_pass = sum(1 for s, _, _ in results if s == 'PASS')
    print(f"결과: {n_pass}/10 PASS")
    if n_pass == 10:
        print("모든 검증 통과 — 개편 완료")
    else:
        fails = [name for s, name, _ in results if s == 'FAIL']
        print(f"실패 항목: {fails}")
    return n_pass == 10


run_final_checks()

---
## 8. 전체 파이프라인 요약

```
S&P500 (SPY)
  ├── 미래 t+1~t+20 집약 (Non-overlapping / Overlapping)
  │       └── RobustScaler → Multi-Variant HMM → regime_label[t] (딥러닝 Y)
  │               (A:full, B:diag, C:GMM, D:overlap v2)
  │
  └── 과거 피처 (Track 1, X) ──┐
                                 ├── Late Fusion (Attention) → 예측
FRED 매크로 (Track 2, X) ───────┘
```

### 출력 파일
| 파일 | 내용 |
|------|------|
| `sp500_regime_dataset_final.csv` | Track1(12) + Track2(6+diff) + regime_label + regime_prob_0..N-1 + close |
| `../output/regime_metadata.json` | best_variant, best_n, event_f1, scaler_params 등 |
